# dbmask — 5-minute browser demo

This notebook demonstrates the complete:

scan → mask → validate

workflow using a small synthetic SQLite database.

No production data is used.
No external LLM is enabled.
Nothing needs to be installed on your own computer.

The notebook installs dbmask from PyPI only inside this temporary
Google Colab runtime.

In [1]:
!pip -q install -U dbmask

import importlib.metadata

print("Installed dbmask version:",
      importlib.metadata.version("dbmask"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.5 MB/s eta 0:00:00
Installed dbmask version: 0.1.2


In [2]:
import sqlite3
import shutil
import pathlib
import os

for filename in [
    "demo.db",
    "demo_original.db",
    "dbmask.yaml",
]:
    if os.path.exists(filename):
        os.remove(filename)

rows = [
    (1, "Mary Johnson", "mary.johnson@corp.example"),
    (2, "Robert Smith", "robert.smith@corp.example"),
    (3, "Linda Davis", "linda.davis@corp.example"),
    (4, "James Wilson", "james.wilson@corp.example"),
    (5, "Patricia Brown", "patricia.brown@corp.example"),
]

with sqlite3.connect("demo.db") as db:
    db.execute("""
        CREATE TABLE customers (
            id INTEGER PRIMARY KEY,
            full_name TEXT,
            email TEXT
        )
    """)

    db.executemany(
        "INSERT INTO customers VALUES (?, ?, ?)",
        rows
    )

    db.commit()

# untouched source copy used by validation
shutil.copyfile("demo.db", "demo_original.db")

config = """
database:
  url: sqlite:///demo.db

source_database:
  url: sqlite:///demo_original.db

detection:
  skip_column_patterns:
    - "^id$"

  # Reduced only because this browser demo uses five synthetic rows.
  pattern_min_samples: 1
  pattern_min_ratio: 0.8

history:
  enabled: false

llm:
  enabled: false

masking:
  seed: dbmask-colab-demo
"""

pathlib.Path("dbmask.yaml").write_text(config)

print("Synthetic demo database created.")
print("No external services or LLMs are enabled.")

Synthetic demo database created.
No external services or LLMs are enabled.


## 1. Discover sensitive columns

`scan` is read-only.

In [3]:
!dbmask scan --config dbmask.yaml

[ok       ] main.customers.id (skip, conf=1.00)
[SENSITIVE] main.customers.full_name -> full_name (pattern, match=100.00%)
  Pattern suggestion 'full_name'; requires review; Distinct nonblank sample evidence: full_name: 5/5 (100.00%)
[SENSITIVE] main.customers.email -> email (pattern, match=100.00%)
  Pattern suggestion 'email'; requires review; Distinct nonblank sample evidence: email: 5/5 (100.00%)

--- Summary ---
Columns analyzed : 3
Sensitive found  : 2
Needs review     : 0 (unknown)
By source        : {'skip': 1, 'pattern': 2}
LLM tokens used  : 0


### What you should see

The scan should identify the synthetic `full_name` and `email`
columns as sensitive, while `id` should remain non-sensitive.

> Exact terminal output may vary slightly by dbmask version.

## 2. Preview masking

No database changes are applied yet.

In [4]:
!dbmask mask --config dbmask.yaml

=== Masking DRY-RUN (no changes written) ===

main.customers  (scanned=5, written=0)
  - full_name: rule=full_name -> strategy=fake_name
  - email: rule=email -> strategy=fake_email
    before: {'full_name': '**** *******', 'email': '****.*******@****.*******'}
    after : {'full_name': 'Amanda Nguyen', 'email': 'donald.sanchez388@example.invalid'}
    before: {'full_name': '****** *****', 'email': '******.*****@****.*******'}
    after : {'full_name': 'Jason Lee', 'email': 'patricia.ramirez183@example.invalid'}
    before: {'full_name': '***** *****', 'email': '*****.*****@****.*******'}
    after : {'full_name': 'Betty Johnson', 'email': 'elizabeth.young225@example.invalid'}

(original values are redacted; pass --show-values to display them)

Re-run with --apply to write these changes back.


## 3. Apply masking

The synthetic target database is now modified.
The untouched source copy remains unchanged.

In [5]:
!dbmask mask --config dbmask.yaml --apply

=== Masking APPLIED ===

main.customers  (scanned=5, written=5)
  - full_name: rule=full_name -> strategy=fake_name
  - email: rule=email -> strategy=fake_email
    before: {'full_name': '**** *******', 'email': '****.*******@****.*******'}
    after : {'full_name': 'Amanda Nguyen', 'email': 'donald.sanchez388@example.invalid'}
    before: {'full_name': '****** *****', 'email': '******.*****@****.*******'}
    after : {'full_name': 'Jason Lee', 'email': 'patricia.ramirez183@example.invalid'}
    before: {'full_name': '***** *****', 'email': '*****.*****@****.*******'}
    after : {'full_name': 'Betty Johnson', 'email': 'elizabeth.young225@example.invalid'}

(original values are redacted; pass --show-values to display them)


## 4. Verify the result

Validation compares the masked target against the untouched source.

In [6]:
!dbmask validate --config dbmask.yaml --strict

[✓] row_count              main.customers: Row counts match (5).
[✓] schema_elements        main.customers: columns/types match.
[✓] schema_elements        main.customers: primary key match.
[✓] schema_elements        main.customers: indexes match.
[✓] schema_elements        main.customers: foreign keys match.
[✓] schema_elements        main.customers: unique constraints match.
[✓] schema_elements        main.customers: check constraints match.
[✓] masking_completeness   main.customers.full_name: PK-aligned comparison of 5 row(s): every sensitive value differs from its source.
[✓] masking_completeness   main.customers.email: PK-aligned comparison of 5 row(s): every sensitive value differs from its source.

--- Validation summary ---
  pass    : 9

RESULT: PASSED ✓


## 5. Inspect the result

The IDs should stay unchanged, while the synthetic names and email
addresses should differ from the original values.

In [7]:
def read_customers(filename):
    with sqlite3.connect(filename) as db:
        return db.execute(
            "SELECT id, full_name, email FROM customers ORDER BY id"
        ).fetchall()

before = read_customers("demo_original.db")
after = read_customers("demo.db")

print("BEFORE MASKING")
for row in before:
    print(row)

print("\nAFTER MASKING")
for row in after:
    print(row)

BEFORE MASKING
(1, 'Mary Johnson', 'mary.johnson@corp.example')
(2, 'Robert Smith', 'robert.smith@corp.example')
(3, 'Linda Davis', 'linda.davis@corp.example')
(4, 'James Wilson', 'james.wilson@corp.example')
(5, 'Patricia Brown', 'patricia.brown@corp.example')

AFTER MASKING
(1, 'Amanda Nguyen', 'donald.sanchez388@example.invalid')
(2, 'Jason Lee', 'patricia.ramirez183@example.invalid')
(3, 'Betty Johnson', 'elizabeth.young225@example.invalid')
(4, 'Matthew Robinson', 'daniel.sanchez322@example.invalid')
(5, 'Stephanie Brown', 'jacob.harris758@example.invalid')


## ✅ Demo complete

If the validation step above shows `RESULT: PASSED`, the demo completed successfully.

You have just run the full dbmask workflow:

**scan → preview → mask → validate**

using only synthetic data in a temporary Google Colab runtime.